<a href="https://colab.research.google.com/github/lidwaa/cinetrack/blob/main/Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
crop_above_table.py
-------------------
Détecte le contour supérieur d'un tableau dans une image de carte grise
espagnole (MDF) et supprime tout ce qui se trouve au-dessus.

Usage:
    python crop_above_table.py input.jpg           -> génère input_cropped.jpg
    python crop_above_table.py input.jpg -o out.jpg
    python crop_above_table.py input.jpg --debug   -> affiche les étapes intermédiaires

Dépendances:
    pip install opencv-python-headless numpy
"""

import cv2
import numpy as np
import argparse
import os
import sys


def detect_table_top(image_path: str, debug: bool = False) -> int:
    """
    Détecte la coordonnée Y du bord supérieur du tableau principal.

    Stratégie :
    1. Pré-traitement : niveaux de gris + flou léger
    2. Détection de lignes horizontales via morphologie (érosion/dilatation)
    3. La PREMIERE ligne horizontale longue = bord supérieur du tableau
    4. Fallback : détection de contours si la morphologie échoue

    Returns:
        y_top (int) : coordonnée Y du bord supérieur du tableau
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Impossible de lire l'image : {image_path}")

    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # --- Étape 1 : binarisation adaptative ---
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    binary = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        15, 4
    )

    # --- Étape 2 : extraction des lignes horizontales ---
    # Noyau large = seules les lignes qui s'étendent sur ≥ 40% de la largeur survivent
    min_line_width = int(w * 0.40)
    kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (min_line_width, 1))
    horizontal_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_h)

    # Dilate légèrement pour connecter les segments proches
    dilate_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3))
    horizontal_lines = cv2.dilate(horizontal_lines, dilate_kernel)

    if debug:
        cv2.imwrite("debug_binary.jpg", binary)
        cv2.imwrite("debug_hlines.jpg", horizontal_lines)

    # --- Étape 3 : trouver la première ligne horizontale ---
    # On somme chaque ligne de pixels : une forte somme = ligne détectée
    row_sums = horizontal_lines.sum(axis=1)

    # Seuil : la ligne doit couvrir au moins 30% de la largeur
    threshold = w * 0.30 * 255

    candidate_rows = np.where(row_sums > threshold)[0]

    if len(candidate_rows) > 0:
        y_top = int(candidate_rows[0])
        print(f"✓ Bord supérieur du tableau détecté à Y = {y_top}px (méthode: lignes horizontales)")
        return y_top

    # --- Fallback : détection de contours rectangulaires ---
    print("⚠ Fallback : recherche de contours rectangulaires...")
    edges = cv2.Canny(blur, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_y = None
    best_score = 0

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        # On cherche un rectangle large (> 50% de w) et pas trop petit (> 5% de h)
        if cw > w * 0.50 and ch > h * 0.05:
            score = cw * ch
            if score > best_score:
                best_score = score
                best_y = y

    if best_y is not None:
        print(f"✓ Bord supérieur détecté à Y = {best_y}px (méthode: contours)")
        return best_y

    # Dernier recours : 20% depuis le haut
    fallback_y = int(h * 0.20)
    print(f"⚠ Aucun tableau trouvé. Crop par défaut à Y = {fallback_y}px (20% depuis le haut)")
    return fallback_y


def crop_above_table(image_path: str, output_path: str = None, debug: bool = False,
                     margin: int = 2) -> str:
    """
    Coupe tout ce qui est au-dessus du bord supérieur du tableau.

    Args:
        image_path  : chemin de l'image source
        output_path : chemin de sortie (par défaut : <nom>_cropped.<ext>)
        debug       : enregistre les images intermédiaires
        margin      : pixels de marge à garder au-dessus de la ligne (défaut 2)

    Returns:
        output_path (str) : chemin de l'image croppée
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Impossible de lire : {image_path}")

    y_top = detect_table_top(image_path, debug=debug)

    # On garde une petite marge pour ne pas couper le filet supérieur du tableau
    y_crop = max(0, y_top - margin)

    cropped = img[y_crop:, :]

    if output_path is None:
        base, ext = os.path.splitext(image_path)
        output_path = f"{base}_cropped{ext}"

    cv2.imwrite(output_path, cropped)

    h_orig = img.shape[0]
    h_crop = cropped.shape[0]
    removed = y_crop
    print(f"✓ Supprimé : {removed}px en haut ({removed/h_orig*100:.1f}% de la hauteur originale)")
    print(f"✓ Image croppée : {h_crop}px de hauteur → sauvegardée dans : {output_path}")

    if debug:
        debug_img = img.copy()
        cv2.line(debug_img, (0, y_top), (img.shape[1], y_top), (0, 0, 255), 3)
        cv2.imwrite("debug_detection.jpg", debug_img)
        print("✓ Image de debug (ligne rouge = bord détecté) : debug_detection.jpg")

    return output_path


# ──────────────────────────────────────────────
# CLI
# ──────────────────────────────────────────────
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Crop tout ce qui est au-dessus du tableau dans une carte grise espagnole."
    )
    parser.add_argument("input", help="Chemin de l'image source (JPG, PNG, TIFF…)")
    parser.add_argument("-o", "--output", default=None,
                        help="Chemin de l'image de sortie (défaut : <nom>_cropped.<ext>)")
    parser.add_argument("--debug", action="store_true",
                        help="Enregistre les images intermédiaires de diagnostic")
    parser.add_argument("--margin", type=int, default=2,
                        help="Pixels de marge à conserver au-dessus du tableau (défaut : 2)")

    args = parser.parse_args()

    if not os.path.exists(args.input):
        print(f"Erreur : fichier introuvable → {args.input}", file=sys.stderr)
        sys.exit(1)

    try:
        out = crop_above_table(args.input, args.output, debug=args.debug, margin=args.margin)
        print(f"\n✅ Terminé ! Image croppée : {out}")
    except Exception as e:
        print(f"Erreur : {e}", file=sys.stderr)
        sys.exit(1)

In [ ]:
"""
crop_carta_gris.py
------------------
Version notebook-friendly : une seule fonction à appeler, tous les paramètres
sont des arguments avec valeurs par défaut.

Dépendances :
    pip install opencv-python pillow numpy

Usage dans un notebook :
    from crop_carta_gris import crop_tabla_superior

    # Minimal
    crop_tabla_superior("input.png", "output.png")

    # Avec réglages
    crop_tabla_superior(
        input_path="input.png",
        output_path="output.png",
        margin=10,
        fallback_ratio=0.35,
        min_line_ratio=0.4,
        deskew_enabled=True,
        deskew_max_angle=5.0,
        canny_low=30,
        canny_high=100,
        search_top_margin=0.15,
        search_bottom_margin=0.90,
        debug=True,
    )
"""

import logging
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
_log = logging.getLogger(__name__)


def crop_tabla_superior(
    input_path: str,
    output_path: str,
    # ── Crop ──────────────────────────────────────────────────────────────────
    margin: int = 5,
    # ── Fallback ──────────────────────────────────────────────────────────────
    fallback_ratio: float = 0.35,
    # ── Détection des lignes ──────────────────────────────────────────────────
    min_line_ratio: float = 0.4,
    canny_low: int = 30,
    canny_high: int = 100,
    search_top_margin: float = 0.15,
    search_bottom_margin: float = 0.90,
    # ── Deskew ────────────────────────────────────────────────────────────────
    deskew_enabled: bool = True,
    deskew_max_angle: float = 5.0,
    # ── Debug ─────────────────────────────────────────────────────────────────
    debug: bool = False,
) -> Image.Image:
    """
    Détecte le bord supérieur du tableau d'une carte grise espagnole et
    retourne (+ sauvegarde) l'image croppée en dessous de ce bord.

    Paramètres
    ----------
    input_path            : chemin de l'image PNG source
    output_path           : chemin de l'image PNG de sortie
    margin                : pixels à garder au-dessus du bord détecté (défaut 5)
    fallback_ratio        : position de crop si détection échoue, fraction de la
                            hauteur totale (défaut 0.35 = 35%)
    min_line_ratio        : fraction de la largeur minimale qu'une ligne doit
                            couvrir pour être candidate (défaut 0.4 = 40%)
    canny_low             : seuil bas du filtre Canny (défaut 30)
    canny_high            : seuil haut du filtre Canny (défaut 100)
    search_top_margin     : fraction haute de l'image ignorée lors de la
                            recherche — évite les filets de l'en-tête (défaut 0.15)
    search_bottom_margin  : fraction basse ignorée lors de la recherche (défaut 0.90)
    deskew_enabled        : active le redressement automatique de l'inclinaison
                            (défaut True)
    deskew_max_angle      : angle max en degrés au-delà duquel on renonce à
                            redresser (défaut 5.0)
    debug                 : sauvegarde une image annotée *_debug.png à côté du
                            fichier de sortie (défaut False)

    Retourne
    --------
    PIL.Image.Image : image croppée, aussi sauvegardée dans output_path
    """

    # ── 1. Chargement ─────────────────────────────────────────────────────────
    img_cv = cv2.imread(input_path)
    if img_cv is None:
        raise ValueError(f"Impossible de lire l'image : {input_path}")
    img_pil = Image.open(input_path).convert("RGB")

    h_orig, w_orig = img_cv.shape[:2]
    _log.info(f"Image chargée : {w_orig}x{h_orig}px — {input_path}")

    # ── 2. Deskew (redressement de l'inclinaison) ─────────────────────────────
    skew_angle = 0.0
    img_cv_work = img_cv.copy()

    if deskew_enabled:
        gray_d = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
        _, binary_d = cv2.threshold(
            gray_d, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )
        kernel_d = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
        dilated_d = cv2.dilate(binary_d, kernel_d)
        coords_d = np.column_stack(np.where(dilated_d > 0))

        if coords_d.shape[0] < 10:
            _log.warning("Deskew : pas assez de pixels détectés, redressement ignoré.")
        else:
            angle_d = cv2.minAreaRect(coords_d)[-1]
            if angle_d < -45:
                angle_d = 90 + angle_d

            if abs(angle_d) > deskew_max_angle:
                _log.warning(
                    f"Deskew : angle détecté ({angle_d:.2f}°) > seuil "
                    f"({deskew_max_angle}°) — redressement ignoré."
                )
            else:
                center_d = (w_orig // 2, h_orig // 2)
                M_d = cv2.getRotationMatrix2D(center_d, angle_d, 1.0)
                img_cv_work = cv2.warpAffine(
                    img_cv, M_d, (w_orig, h_orig),
                    flags=cv2.INTER_CUBIC,
                    borderMode=cv2.BORDER_REPLICATE,
                )
                skew_angle = angle_d
                if abs(skew_angle) > 0.1:
                    img_pil = img_pil.rotate(
                        -skew_angle, resample=Image.BICUBIC, expand=False
                    )
                _log.info(f"Deskew : image redressée de {skew_angle:.2f}°")

    # ── 3. Détection du bord supérieur du tableau ─────────────────────────────
    gray = cv2.cvtColor(img_cv_work, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, canny_low, canny_high, apertureSize=3)

    min_length = int(w * min_line_ratio)
    _log.info(
        f"Détection : longueur min de ligne = {min_length}px "
        f"({min_line_ratio * 100:.0f}% de {w}px)"
    )

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=80,
        minLineLength=min_length,
        maxLineGap=20,
    )

    top_y = None
    fallback_used = False

    if lines is not None:
        upper_bound = int(h * search_top_margin)
        lower_bound = int(h * search_bottom_margin)
        horizontal_ys = []

        for line in lines:
            x1, y1, x2, y2 = line[0]
            angle_l = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if angle_l < 3.0:
                horizontal_ys.append(min(y1, y2))

        candidates = [y for y in horizontal_ys if upper_bound < y < lower_bound]

        if candidates:
            top_y = min(candidates)
            _log.info(f"Bord supérieur du tableau détecté à Y = {top_y}px")
        else:
            _log.warning("Aucune ligne candidate dans la zone de recherche.")
    else:
        _log.warning("HoughLinesP : aucune ligne détectée.")

    # ── 4. Fallback si détection échoue ───────────────────────────────────────
    if top_y is None:
        fallback_y = int(h_orig * fallback_ratio)
        _log.warning(
            f"Détection échouée → fallback : crop à Y={fallback_y} "
            f"({fallback_ratio * 100:.0f}% de la hauteur)"
        )
        top_y = fallback_y
        fallback_used = True

    # ── 5. Image de debug ─────────────────────────────────────────────────────
    if debug:
        debug_img = img_cv_work.copy()
        dh, dw = debug_img.shape[:2]
        color = (0, 165, 255) if fallback_used else (0, 0, 255)
        label = f"FALLBACK Y={top_y}" if fallback_used else f"Table top Y={top_y}"
        cv2.line(debug_img, (0, top_y), (dw, top_y), color, 3)
        cv2.putText(
            debug_img, label,
            (10, max(top_y - 10, 40)),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2,
        )
        debug_path = Path(output_path).with_suffix("").as_posix() + "_debug.png"
        cv2.imwrite(debug_path, debug_img)
        _log.info(f"Debug image sauvegardée : {debug_path}")

    # ── 6. Crop ───────────────────────────────────────────────────────────────
    w_pil, h_pil = img_pil.size
    crop_y = max(0, top_y - margin)
    _log.info(f"Crop : de Y={crop_y} à Y={h_pil} (marge={margin}px)")
    result = img_pil.crop((0, crop_y, w_pil, h_pil))

    # ── 7. Sauvegarde ─────────────────────────────────────────────────────────
    result.save(output_path, format="PNG")
    w_out, h_out = result.size
    _log.info(f"Image sauvegardée : {output_path} ({w_out}x{h_out}px)")

    if fallback_used:
        _log.warning("⚠️  Fallback utilisé — vérifiez le résultat manuellement.")
    else:
        _log.info("✅ Crop réussi via détection automatique.")

    return result

In [ ]:
# Cellule 1 — installation
# !pip install opencv-python pillow numpy

# Cellule 2 — import
from crop_carta_gris import crop_tabla_superior

# Cellule 3 — appel minimal
result = crop_tabla_superior("input.png", "output.png")

# Cellule 4 — affichage inline (optionnel)
display(result)

In [ ]:
"""
crop_carta_gris.py
------------------
Version notebook-friendly : une seule fonction à appeler, tous les paramètres
sont des arguments avec valeurs par défaut.

Dépendances :
    pip install opencv-python pillow numpy

Usage dans un notebook :
    from crop_carta_gris import crop_tabla_superior

    # Minimal
    crop_tabla_superior("input.png", "output.png")

    # Avec réglages
    crop_tabla_superior(
        input_path="input.png",
        output_path="output.png",
        margin=10,
        fallback_ratio=0.35,
        min_line_ratio=0.4,
        deskew_enabled=True,
        deskew_max_angle=5.0,
        deskew_zone_start=0.30,   # deskew calculé uniquement sur les 30%-90% de l'image
        deskew_zone_end=0.90,     # (zone du tableau, hors logo/numéro de série)
        canny_low=30,
        canny_high=100,
        search_top_margin=0.15,
        search_bottom_margin=0.90,
        debug=True,
    )
"""

import logging
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
_log = logging.getLogger(__name__)


def crop_tabla_superior(
    input_path: str,
    output_path: str,
    # ── Crop ──────────────────────────────────────────────────────────────────
    margin: int = 5,
    # ── Fallback ──────────────────────────────────────────────────────────────
    fallback_ratio: float = 0.35,
    # ── Détection des lignes ──────────────────────────────────────────────────
    min_line_ratio: float = 0.4,
    canny_low: int = 30,
    canny_high: int = 100,
    search_top_margin: float = 0.15,
    search_bottom_margin: float = 0.90,
    # ── Deskew ────────────────────────────────────────────────────────────────
    deskew_enabled: bool = True,
    deskew_max_angle: float = 5.0,
    deskew_zone_start: float = 0.30,
    deskew_zone_end: float = 0.90,
    # ── Debug ─────────────────────────────────────────────────────────────────
    debug: bool = False,
) -> Image.Image:
    """
    Détecte le bord supérieur du tableau d'une carte grise espagnole et
    retourne (+ sauvegarde) l'image croppée en dessous de ce bord.

    Paramètres
    ----------
    input_path            : chemin de l'image PNG source
    output_path           : chemin de l'image PNG de sortie
    margin                : pixels à garder au-dessus du bord détecté (défaut 5)
    fallback_ratio        : position de crop si détection échoue, fraction de la
                            hauteur totale (défaut 0.35 = 35%)
    min_line_ratio        : fraction de la largeur minimale qu'une ligne doit
                            couvrir pour être candidate (défaut 0.4 = 40%)
    canny_low             : seuil bas du filtre Canny (défaut 30)
    canny_high            : seuil haut du filtre Canny (défaut 100)
    search_top_margin     : fraction haute de l'image ignorée lors de la
                            recherche — évite les filets de l'en-tête (défaut 0.15)
    search_bottom_margin  : fraction basse ignorée lors de la recherche (défaut 0.90)
    deskew_enabled        : active le redressement automatique de l'inclinaison
                            (défaut True)
    deskew_max_angle      : angle max en degrés au-delà duquel on renonce à
                            redresser (défaut 5.0)
    deskew_zone_start     : fraction Y de début de la zone utilisée pour calculer
                            l'angle — exclut le logo/numéro de série en haut
                            (défaut 0.30 = à partir de 30% de la hauteur)
    deskew_zone_end       : fraction Y de fin de la zone de calcul d'angle
                            (défaut 0.90)
    debug                 : sauvegarde une image annotée *_debug.png à côté du
                            fichier de sortie (défaut False)

    Retourne
    --------
    PIL.Image.Image : image croppée, aussi sauvegardée dans output_path
    """

    # ── 1. Chargement ─────────────────────────────────────────────────────────
    img_cv = cv2.imread(input_path)
    if img_cv is None:
        raise ValueError(f"Impossible de lire l'image : {input_path}")
    img_pil = Image.open(input_path).convert("RGB")

    h_orig, w_orig = img_cv.shape[:2]
    _log.info(f"Image chargée : {w_orig}x{h_orig}px — {input_path}")

    # ── 2. Deskew restreint à la zone tableau (hors logo / numéro de série) ───
    #
    #  Stratégie :
    #    On isole une bande verticale [deskew_zone_start .. deskew_zone_end]
    #    qui correspond approximativement à la zone du tableau, bien en dessous
    #    du logo et du numéro de série. L'angle est calculé uniquement sur cette
    #    bande, puis appliqué à l'image entière.
    #    → Le logo et le numéro de série ne peuvent plus biaiser l'angle.
    #
    skew_angle = 0.0
    img_cv_work = img_cv.copy()

    if deskew_enabled:
        y_start_d = int(h_orig * deskew_zone_start)
        y_end_d   = int(h_orig * deskew_zone_end)
        zone_d    = img_cv[y_start_d:y_end_d, :]

        _log.info(
            f"Deskew : calcul de l'angle sur la bande Y=[{y_start_d}, {y_end_d}] "
            f"({deskew_zone_start*100:.0f}%–{deskew_zone_end*100:.0f}% de la hauteur)"
        )

        gray_d = cv2.cvtColor(zone_d, cv2.COLOR_BGR2GRAY)

        # Binarisation Otsu — robuste aux variations de luminosité du scan
        _, binary_d = cv2.threshold(
            gray_d, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # Dilatation horizontale : renforce les lignes du tableau,
        # écrase le texte court (numéros dans les cellules)
        kernel_d = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
        dilated_d = cv2.dilate(binary_d, kernel_d)

        coords_d = np.column_stack(np.where(dilated_d > 0))

        if coords_d.shape[0] < 10:
            _log.warning("Deskew : pas assez de pixels dans la zone tableau, redressement ignoré.")
        else:
            angle_d = cv2.minAreaRect(coords_d)[-1]
            # minAreaRect retourne un angle dans [-90, 0) → on normalise
            if angle_d < -45:
                angle_d = 90 + angle_d

            if abs(angle_d) > deskew_max_angle:
                _log.warning(
                    f"Deskew : angle détecté ({angle_d:.2f}°) > seuil "
                    f"({deskew_max_angle}°) — redressement ignoré."
                )
            else:
                center_d = (w_orig // 2, h_orig // 2)
                M_d = cv2.getRotationMatrix2D(center_d, angle_d, 1.0)
                img_cv_work = cv2.warpAffine(
                    img_cv, M_d, (w_orig, h_orig),
                    flags=cv2.INTER_CUBIC,
                    borderMode=cv2.BORDER_REPLICATE,
                )
                skew_angle = angle_d
                if abs(skew_angle) > 0.1:
                    img_pil = img_pil.rotate(
                        -skew_angle, resample=Image.BICUBIC, expand=False
                    )
                _log.info(f"Deskew : image redressée de {skew_angle:.2f}°")

    # ── 3. Détection du bord supérieur du tableau (Canny + Hough) ─────────────
    gray = cv2.cvtColor(img_cv_work, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges   = cv2.Canny(blurred, canny_low, canny_high, apertureSize=3)

    min_length = int(w * min_line_ratio)
    _log.info(
        f"Détection : longueur min de ligne = {min_length}px "
        f"({min_line_ratio * 100:.0f}% de {w}px)"
    )

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=80,
        minLineLength=min_length,
        maxLineGap=20,
    )

    top_y        = None
    fallback_used = False

    if lines is not None:
        upper_bound  = int(h * search_top_margin)
        lower_bound  = int(h * search_bottom_margin)
        horizontal_ys = []

        for line in lines:
            x1, y1, x2, y2 = line[0]
            angle_l = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if angle_l < 3.0:
                horizontal_ys.append(min(y1, y2))

        candidates = [y for y in horizontal_ys if upper_bound < y < lower_bound]

        if candidates:
            top_y = min(candidates)
            _log.info(f"Bord supérieur du tableau détecté à Y = {top_y}px")
        else:
            _log.warning("Aucune ligne candidate dans la zone de recherche.")
    else:
        _log.warning("HoughLinesP : aucune ligne détectée.")

    # ── 4. Fallback si détection échoue ───────────────────────────────────────
    if top_y is None:
        fallback_y = int(h_orig * fallback_ratio)
        _log.warning(
            f"Détection échouée → fallback : crop à Y={fallback_y} "
            f"({fallback_ratio * 100:.0f}% de la hauteur)"
        )
        top_y        = fallback_y
        fallback_used = True

    # ── 5. Image de debug ─────────────────────────────────────────────────────
    if debug:
        debug_img = img_cv_work.copy()
        dh, dw    = debug_img.shape[:2]

        # Ligne de crop (rouge = détection, orange = fallback)
        color = (0, 165, 255) if fallback_used else (0, 0, 255)
        label = f"FALLBACK Y={top_y}" if fallback_used else f"Table top Y={top_y}"
        cv2.line(debug_img, (0, top_y), (dw, top_y), color, 3)
        cv2.putText(
            debug_img, label,
            (10, max(top_y - 10, 40)),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2,
        )

        # Zone utilisée pour le deskew (rectangle vert semi-transparent)
        if deskew_enabled:
            y_start_viz = int(dh * deskew_zone_start)
            y_end_viz   = int(dh * deskew_zone_end)
            overlay = debug_img.copy()
            cv2.rectangle(overlay, (0, y_start_viz), (dw, y_end_viz), (0, 200, 0), -1)
            cv2.addWeighted(overlay, 0.08, debug_img, 0.92, 0, debug_img)
            cv2.rectangle(debug_img, (0, y_start_viz), (dw, y_end_viz), (0, 200, 0), 2)
            cv2.putText(
                debug_img, "deskew zone",
                (10, y_start_viz + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2,
            )

        debug_path = Path(output_path).with_suffix("").as_posix() + "_debug.png"
        cv2.imwrite(debug_path, debug_img)
        _log.info(f"Debug image sauvegardée : {debug_path}")

    # ── 6. Crop ───────────────────────────────────────────────────────────────
    w_pil, h_pil = img_pil.size
    crop_y  = max(0, top_y - margin)
    _log.info(f"Crop : de Y={crop_y} à Y={h_pil} (marge={margin}px)")
    result  = img_pil.crop((0, crop_y, w_pil, h_pil))

    # ── 7. Sauvegarde ─────────────────────────────────────────────────────────
    result.save(output_path, format="PNG")
    w_out, h_out = result.size
    _log.info(f"Image sauvegardée : {output_path} ({w_out}x{h_out}px)")

    if fallback_used:
        _log.warning("⚠️  Fallback utilisé — vérifiez le résultat manuellement.")
    else:
        _log.info("✅ Crop réussi via détection automatique.")

    return result